# 🚗 YOLOv8n — Training Klasifikasi Golongan Kendaraan Tol

**SEBELUM MULAI:** Aktifkan GPU dulu!
```
Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save
```

| Golongan | Keterangan |
|----------|------------|
| GOL I    | Sedan / Jip / Pick-up / Bus |
| GOL II   | Truk 2 Gandar |
| GOL III  | Truk 3 Gandar |
| GOL IV   | Truk 4 Gandar |
| GOL V    | Truk 5 Gandar atau lebih |

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Cek GPU
# ═══════════════════════════════════════════════════════════════
import torch

print('=' * 55)
if torch.cuda.is_available():
    print(f'  ✅ GPU aktif  : {torch.cuda.get_device_name(0)}')
    print(f'  💾 VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('  ❌ GPU TIDAK AKTIF!')
    print('  → Runtime → Change runtime type → T4 GPU → Save')
    raise SystemExit('Aktifkan GPU dulu sebelum lanjut!')
print('=' * 55)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Install library
# ═══════════════════════════════════════════════════════════════
!pip install ultralytics roboflow -q
print('✅ ultralytics & roboflow terinstall')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Download Dataset dari Roboflow
# Ganti API_KEY dengan key dari: https://app.roboflow.com → Settings → API Keys
# ═══════════════════════════════════════════════════════════════
from roboflow import Roboflow

API_KEY = 'GANTI_DENGAN_API_KEY_KAMU'   # ← WAJIB GANTI!

if API_KEY == 'GANTI_DENGAN_API_KEY_KAMU':
    raise ValueError('Ganti API_KEY dengan key Roboflow kamu!')

rf      = Roboflow(api_key=API_KEY)
project = rf.workspace('muhammad-rizky-ferdiansyah-00fow').project(
    'golongan-kendaraan-jalan-tol-lingkar-luar-jakarta-timur'
)
dataset = project.version(5).download('yolov8')

print(f'\n✅ Dataset downloaded ke: {dataset.location}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Buat data.yaml
# ═══════════════════════════════════════════════════════════════
import yaml, os
from pathlib import Path

DATASET_DIR = Path(dataset.location)
YAML_PATH   = '/content/data.yaml'

cfg = {
    'train': str(DATASET_DIR / 'train' / 'images'),
    'val':   str(DATASET_DIR / 'valid' / 'images'),
    'test':  str(DATASET_DIR / 'test'  / 'images'),
    'nc': 5,
    'names': ['GOL I', 'GOL II', 'GOL III', 'GOL IV', 'GOL V'],
}

with open(YAML_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('✅ data.yaml dibuat:')
print(open(YAML_PATH).read())

print('Jumlah gambar per split:')
for split in ['train', 'valid', 'test']:
    p = DATASET_DIR / split / 'images'
    n = len(list(p.glob('*.[jJpP][pPnN][gG]*'))) if p.exists() else 0
    print(f'  {split:6s}: {n} gambar')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — TRAINING  (⏳ 1–2 jam, jangan tutup tab!)
# ═══════════════════════════════════════════════════════════════
from ultralytics import YOLO
import time

YAML_PATH = '/content/data.yaml'   # didefinisikan ulang agar cell mandiri

print('=' * 60)
print('  YOLOv8n — Training Golongan Kendaraan Tol')
print('  GPU: ' + __import__('torch').cuda.get_device_name(0))
print('=' * 60)

model = YOLO('yolov8n.pt')
t0    = time.time()

results = model.train(
    data        = YAML_PATH,
    epochs      = 100,
    imgsz       = 640,
    batch       = 16,        # GPU T4: 16 aman
    device      = 0,
    workers     = 2,
    project     = '/content/runs',
    name        = 'vehicle_cls_v1',
    exist_ok    = True,
    patience    = 20,
    save        = True,
    save_period = 10,
    plots       = True,
    verbose     = True,
    # Augmentasi Paper Section 3.3
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    degrees     = 0.0,
    translate   = 0.1,
    scale       = 0.5,
    shear       = 0.0,
    perspective = 0.0,
    flipud      = 0.0,
    fliplr      = 0.5,
    mosaic      = 1.0,
    mixup       = 0.0,
    copy_paste  = 0.0,
    erasing     = 0.4,
)

elapsed = time.time() - t0
h, m    = divmod(int(elapsed), 3600)
m, s    = divmod(m, 60)
SAVE_DIR = str(results.save_dir)

print(f'\n✅ Training selesai! {h}j {m}m {s}d')
print(f'📁 Disimpan di: {SAVE_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Evaluasi Metrik  (self-contained)
# ═══════════════════════════════════════════════════════════════
from ultralytics import YOLO

SAVE_DIR  = '/content/runs/vehicle_cls_v1'
YAML_PATH = '/content/data.yaml'
best_pt   = f'{SAVE_DIR}/weights/best.pt'

model   = YOLO(best_pt)
metrics = model.val(data=YAML_PATH, device=0, verbose=False)

CLASS_NAMES = ['GOL I', 'GOL II', 'GOL III', 'GOL IV', 'GOL V']

print('\n' + '=' * 55)
print('  HASIL EVALUASI')
print('=' * 55)
print(f'  mAP@0.50      : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
print(f'  mAP@0.50:0.95 : {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
print(f'  Precision     : {metrics.box.mp:.4f}')
print(f'  Recall        : {metrics.box.mr:.4f}')
print(f'\n  {"Kelas":<12} {"AP@50":>10}  {"AP@50-95":>12}')
print(f'  {"-"*38}')
for i, cls in enumerate(CLASS_NAMES):
    a50 = float(metrics.box.ap50[i]) if i < len(metrics.box.ap50) else 0
    a95 = float(metrics.box.ap[i])   if i < len(metrics.box.ap)   else 0
    print(f'  {cls:<12} {a50:>10.4f}  {a95:>12.4f}')
print('=' * 55)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Export ke ONNX  (self-contained)
# ═══════════════════════════════════════════════════════════════
from ultralytics import YOLO

SAVE_DIR = '/content/runs/vehicle_cls_v1'
best_pt  = f'{SAVE_DIR}/weights/best.pt'

model     = YOLO(best_pt)
onnx_path = model.export(
    format   = 'onnx',
    imgsz    = 640,
    opset    = 12,
    simplify = True,
    device   = 0,
)
print(f'✅ ONNX disimpan: {onnx_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Download Hasil  (self-contained, semua bug sudah fix)
# ═══════════════════════════════════════════════════════════════
import shutil, os, glob
from google.colab import files

# Auto-detect path hasil training
best_pts = glob.glob('/content/**/weights/best.pt', recursive=True)

if not best_pts:
    print('❌ best.pt belum ada!')
    print('   → Pastikan Cell 5 sudah selesai terlebih dahulu.')
    os.system('find /content/runs -type f 2>/dev/null || echo "Folder runs belum ada"')
else:
    SAVE_DIR = os.path.dirname(os.path.dirname(best_pts[0]))
    zip_out  = '/content/vehicle_cls_v1_results.zip'

    print(f'✅ Model ditemukan di: {SAVE_DIR}')
    print('\n📦 File tersedia:')
    for fname in ['best.pt', 'last.pt', 'best.onnx']:
        fp = os.path.join(SAVE_DIR, 'weights', fname)
        if os.path.exists(fp):
            print(f'   ✅ weights/{fname}  ({os.path.getsize(fp)/1e6:.1f} MB)')
        else:
            print(f'   ⚠️  weights/{fname}  tidak ada')

    print('\n📦 Membuat zip ...')
    shutil.make_archive('/content/vehicle_cls_v1_results', 'zip', SAVE_DIR)

    zip_size = os.path.getsize(zip_out) / 1e6
    print(f'⬇️  Mendownload zip ({zip_size:.1f} MB) ...')
    files.download(zip_out)

## 📋 Setelah Download

1. Extract zip hasil download
2. Salin file ke proyek lokal:
   - `weights/best.pt` → `backend/models/best.pt`
   - `weights/best.onnx` → `backend/models/best.onnx`
3. Jalankan backend:
   ```bash
   cd backend
   venv\Scripts\activate
   python main.py
   ```